In [ ]:
from sqlalchemy import create_engine, text
import pyodbc, os, json
import pandas as pd
from dotenv import load_dotenv
import openai
# Load environment variables from .env file
load_dotenv()

#Refer https://knowledgebase.apexsql.com/configure-remote-access-connect-remote-sql-server-instance-apexsql-tools/

True

In [2]:
conn_str = os.getenv('SQL_CONNECTION_STRING')
cnxn = pyodbc.connect(conn_str)

In [4]:
cursor = cnxn.cursor()

# Execute a SELECT query
try:
    # Example: Select all customers
    cursor.execute("SELECT TOP 5 * FROM Customers")
    
    # Fetch and print results
    rows = cursor.fetchall()
    for row in rows:
        print(row)
    
    # If you want to get results as a pandas DataFrame instead:
    df = pd.read_sql_query("SELECT TOP 5 * FROM Customers", cnxn)
    display(df)  # This will show the results in a nice table format in Jupyter

except Exception as e:
    print(f"Error executing query: {e}")
finally:
    # Close the cursor and connection
    cursor.close()
    cnxn.close()

('ALFKI', 'Alfreds Futterkiste', 'Maria Anders', 'Sales Representative', 'Obere Str. 57', 'Berlin', None, '12209', 'Germany', '030-0074321', '030-0076545')
('ANATR', 'Ana Trujillo Emparedados y helados', 'Ana Trujillo', 'Owner', 'Avda. de la Constitución 2222', 'México D.F.', None, '05021', 'Mexico', '(5) 555-4729', '(5) 555-3745')
('ANTON', 'Antonio Moreno Taquería', 'Antonio Moreno', 'Owner', 'Mataderos  2312', 'México D.F.', None, '05023', 'Mexico', '(5) 555-3932', None)
('AROUT', 'Around the Horn', 'Thomas Hardy', 'Sales Representative', '120 Hanover Sq.', 'London', None, 'WA1 1DP', 'UK', '(171) 555-7788', '(171) 555-6750')
('BERGS', 'Berglunds snabbköp', 'Christina Berglund', 'Order Administrator', 'Berguvsvägen  8', 'Luleå', None, 'S-958 22', 'Sweden', '0921-12 34 65', '0921-12 34 67')


/tmp/ipykernel_1222/2015768944.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT TOP 5 * FROM Customers", cnxn)


,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,None,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,None,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,None,05023,Mexico,(5) 555-3932,None
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,None,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,None,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


In [21]:
# Set your API key
#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Query types to generate
query_types = [
    "a simple SELECT",
    "filtering with WHERE conditions",
    "aggregation (COUNT, SUM, AVG, etc.)",
    "multiple table JOINs",
    "sorting with ORDER BY",
    "grouping with GROUP BY",
    "filtering aggregated results with HAVING",
#    "subqueries",
#    "common table expressions (WITH clause)"
]

def clean_sql_response(content: str) -> str:
    """Clean up the SQL response to make it valid JSON."""
    # Remove backslash line continuations and convert to proper newlines
    content = content.replace('\\\n', ' ')
    content = content.replace('\\', '')
    return content

def generate_example(query_type):
    # Create the prompt with the specific query type
    prompt_template = """
    You are an expert SQL developer helping to create training data for a text-to-SQL system. 
    
    DATABASE SCHEMA:
    The database contains the following tables:
    
    1. Customers
       - CustomerID (text, primary key)
       - CompanyName (text)
       - ContactName (text)
       - ContactTitle (text)
       - Address (text)
       - City (text)
       - Region (text)
       - PostalCode (text)
       - Country (text)
       - Phone (text)
       - Fax (text)
    
    2. Orders
       - OrderID (integer, primary key)
       - CustomerID (text, foreign key to Customers.CustomerID)
       - EmployeeID (integer)
       - OrderDate (date)
       - RequiredDate (date)
       - ShippedDate (date)
       - ShipVia (integer)
       - Freight (numeric)
       - ShipName (text)
       - ShipAddress (text)
       - ShipCity (text)
       - ShipRegion (text)
       - ShipPostalCode (text)
       - ShipCountry (text)
    
    3. OrderDetails
       - OrderID (integer, foreign key to Orders.OrderID, part of composite primary key)
       - ProductID (integer, foreign key to Products.ProductID, part of composite primary key)
       - UnitPrice (numeric)
       - Quantity (integer)
       - Discount (real)
    
    4. Products
       - ProductID (integer, primary key)
       - ProductName (text)
       - SupplierID (integer)
       - CategoryID (integer, foreign key to Categories.CategoryID)
       - QuantityPerUnit (text)
       - UnitPrice (numeric)
       - UnitsInStock (integer)
       - UnitsOnOrder (integer)
       - ReorderLevel (integer)
       - Discontinued (integer)
    
    5. Categories
       - CategoryID (integer, primary key)
       - CategoryName (text)
       - Description (text)
       - Picture (binary)
    
    RELATIONSHIPS:
    - Customers have many Orders (CustomerID)
    - Orders have many OrderDetails (OrderID)
    - Products have many OrderDetails (ProductID)
    - Categories have many Products (CategoryID)
    
    TASK:
    Generate 15 unique examples of natural language questions about this database and the corresponding MS SQL query that correctly answers those questions.
    The natural language questions should be something a business user might ask, and the SQL query should follow MS SQL syntax.

    The questions should cover diverse business scenarios such as:
    - Customer Analysis:
        * Customer demographics and locations
        * Customer order history and preferences
        * Top customers by order volume or revenue
        * Customer contact information lookups
    
    - Sales Analysis:
        * Sales trends over time
        * Revenue by product/category
        * Order volumes and frequencies
        * Shipping patterns and delivery times
    
    - Product Analysis:
        * Product inventory levels
        * Popular products and categories
        * Product pricing analysis
        * Discontinued products
    
    - Geographic Analysis:
        * Sales by region/country
        * Customer distribution
        * Shipping destinations
        * Regional performance
    
    - Operational Metrics:
        * Order processing times
        * Shipping efficiency
        * Employee performance
        * Inventory management
    
    Format your response as a list of JSON objects with "input" and "output" fields:
    
    [
      {{
        "input": "Your first natural language question here",
        "output": "Your first MS SQL query here"
      }},
      {{
        "input": "Your second natural language question here",
        "output": "Your second MS SQL query here"
      }}
    ]
    
    Make sure your query:
    1. Uses the correct MS SQL syntax
    2. Uses the exact table and column names from the schema
    3. Does not include any newlines or backslashes, just spaces
    4. Includes JOINs where necessary
    5. Uses aliases for readability when appropriate
    6. Is optimized for better performance
    7. Ensure each query is unique and not similar to the others
    
    Generate 15 questions that require {query_type} in the query. Ensure final output is a list with no ```json 
    """
    
    prompt = prompt_template.format(query_type=query_type)
    
    # Call the API
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates SQL examples."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.95,
        max_tokens=1000
    )
    
    # Extract the JSON from the response
    content = response.choices[0].message.content.strip()
    print(content)

    
    # Extract and clean the JSON from the response
    content = response.choices[0].message.content.strip()
    try:
        cleaned_content = clean_sql_response(content)
        result = json.loads(cleaned_content)
        
            # Save the examples to a file
        with open("synthetic_examples.json", "w") as f:
            json.dump(result, f)
        return result
    except json.JSONDecodeError as e:
        print("Original content:", content)
        print("Cleaned content:", cleaned_content)
        print("JSON Error:", str(e))
        return {"question": "", "query": ""}


        
    return example

In [22]:
r = generate_example(query_types[2])

[
  {
    "input": "How many orders have been placed by each customer?",
    "output": "SELECT CustomerID, COUNT(OrderID) AS OrderCount FROM Orders GROUP BY CustomerID"
  },
  {
    "input": "What is the total revenue generated from all orders?",
    "output": "SELECT SUM(UnitPrice * Quantity - Discount) AS TotalRevenue FROM OrderDetails"
  },
  {
    "input": "Which category has the highest average unit price?",
    "output": "SELECT CategoryID, AVG(UnitPrice) AS AvgUnitPrice FROM Products GROUP BY CategoryID ORDER BY AvgUnitPrice DESC"
  },
  {
    "input": "How many units of each product are currently in stock?",
    "output": "SELECT ProductID, SUM(UnitsInStock) AS TotalUnitsInStock FROM Products GROUP BY ProductID"
  },
  {
    "input": "What is the total freight cost for each order placed by customers in the 'Germany' region?",
    "output": "SELECT o.OrderID, SUM(o.Freight) AS TotalFreightCost FROM Orders o JOIN Customers c ON o.CustomerID = c.CustomerID WHERE c.Country = 'Germa

In [7]:
from src.sql.sql_executor import SQLExecutor

sql_executor = SQLExecutor(conn_str)

sql_executor.validate_query(r['output'])



SQL db connection successful!


True

In [ ]:
# cnxn = pyodbc.connect(conn_str)
# cursor = cnxn.cursor()

# cursor.execute('select * from Customers top 5')

# cursor.fetchall()

ProgrammingError: ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Incorrect syntax near the keyword 'top'. (156) (SQLExecDirectW)")

In [ ]:
# def normalize_sql(sql):
#     """Normalize SQL formatting"""
#     # Remove extra whitespace
#     sql = ' '.join(sql.split())
    
#     # Add newlines after common SQL clauses
#     sql = sql.replace(' SELECT ', '\nSELECT ')
#     sql = sql.replace(' FROM ', '\nFROM ')
#     sql = sql.replace(' WHERE ', '\nWHERE ')
#     sql = sql.replace(' JOIN ', '\nJOIN ')
#     sql = sql.replace(' GROUP BY ', '\nGROUP BY ')
#     sql = sql.replace(' HAVING ', '\nHAVING ')
#     sql = sql.replace(' ORDER BY ', '\nORDER BY ')
#     sql = sql.replace(' LIMIT ', '\nLIMIT ')
    
#     # Remove the first newline if it exists
#     if sql.startswith('\n'):
#         sql = sql[1:]
    
#     return sql

# # Apply to your examples
# for example in examples:
#     example['output'] = normalize_sql(example['output'])